# 12 Cleaning data with pandas

Real data arrive with duplicates, typos, mixed case, impossible values and text
where numbers should be. This notebook cleans `isolates_raw.csv` step by step.

In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv("../data/isolates_raw.csv")
print(raw.shape)
raw.head()

(408, 9)


,isolate_id,patient_id,collected,specimen,organism,ciprofloxacin,ceftriaxone,mic_cipro,temp_c
0,ISO-0001,P0114,26/03/2025,BLOOD,E. coli,R,S,<=0.06,36.8
1,ISO-0002,P0212,10/04/2025,urine,P. aeruginosa,R,R,2,37.0
2,ISO-0003,P0019,24/06/2025,BLOOD,E. coli,S,S,0.5,37.6
3,ISO-0004,P0097,19/05/2025,Wound swab,S. aureus,R,R,0.25,37.9
4,ISO-0005,P0038,18/02/2025,Wound swab,K. pneumoniae,S,S,1,38.6


In [2]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 408 entries, 0 to 407
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   isolate_id     408 non-null    str    
 1   patient_id     408 non-null    str    
 2   collected      408 non-null    str    
 3   specimen       408 non-null    str    
 4   organism       408 non-null    str    
 5   ciprofloxacin  383 non-null    str    
 6   ceftriaxone    376 non-null    str    
 7   mic_cipro      408 non-null    str    
 8   temp_c         408 non-null    float64
dtypes: float64(1), str(8)
memory usage: 28.8 KB


## 1. Duplicates

In [3]:
print(raw.duplicated().sum(), "fully duplicated rows")
print(raw["isolate_id"].duplicated().sum(), "repeated isolate IDs")

iso = raw.drop_duplicates().copy()
print(iso.shape)

8 fully duplicated rows
8 repeated isolate IDs
(400, 9)


## 2. Text: spaces and case

In [4]:
print(iso["specimen"].value_counts())

specimen
BLOOD         82
Wound swab    72
Stool         68
Blood         67
urine         59
Urine         52
Name: count, dtype: int64


In [5]:
iso["specimen"] = iso["specimen"].str.strip().str.capitalize()
iso["specimen"].value_counts()

specimen
Blood         149
Urine         111
Wound swab     72
Stool          68
Name: count, dtype: int64

The `.str` accessor gives every string method from notebook 02, applied to the whole column.

## 3. Consistent categories

In [6]:
for col in ["ciprofloxacin", "ceftriaxone"]:
    print(col, iso[col].value_counts(dropna=False).to_dict())

ciprofloxacin {'S': 173, 'R': 141, nan: 24, 'I': 21, 'r': 21, 's': 20}
ceftriaxone {'S': 186, 'R': 161, nan: 31, 'I': 22}


In [7]:
for col in ["ciprofloxacin", "ceftriaxone"]:
    iso[col] = (
        iso[col]
        .str.strip()
        .str.upper()
        .replace({"N/A": np.nan, "": np.nan})
    )
    iso[col] = pd.Categorical(iso[col], categories=["S", "I", "R"], ordered=True)

iso["ciprofloxacin"].value_counts(dropna=False)

ciprofloxacin
S      193
R      162
NaN     24
I       21
Name: count, dtype: int64

## 4. Dates

Dates were typed as day/month/year. Tell pandas the format rather than letting it guess.

In [8]:
iso["collected"] = pd.to_datetime(iso["collected"], format="%d/%m/%Y")
iso["month"] = iso["collected"].dt.to_period("M")
iso["weekday"] = iso["collected"].dt.day_name()
iso[["collected", "month", "weekday"]].head()

,collected,month,weekday
0,2025-03-26,2025-03,Wednesday
1,2025-04-10,2025-04,Thursday
2,2025-06-24,2025-06,Tuesday
3,2025-05-19,2025-05,Monday
4,2025-02-18,2025-02,Tuesday


## 5. Numbers stored as text

MIC values contain `>4`, `<=0.06` and `not done`. Keep the sign in its own column
and turn the rest into a number.

In [9]:
mic = iso["mic_cipro"].str.strip()
iso["mic_sign"] = mic.str.extract(r"^([<>]=?)", expand=False)
iso["mic_value"] = pd.to_numeric(mic.str.replace(r"^[<>]=?", "", regex=True), errors="coerce")
iso[["mic_cipro", "mic_sign", "mic_value"]].drop_duplicates().sort_values("mic_value")

,mic_cipro,mic_sign,mic_value
0,<=0.06,<=,0.06
3,0.25,NaN,0.25
2,0.5,NaN,0.50
4,1,NaN,1.00
1,2,NaN,2.00
13,>4,>,4.00
6,not done,NaN,NaN


`errors="coerce"` turns anything that is not a number into `NaN` instead of failing.

## 6. Impossible values

In [10]:
print(iso["temp_c"].describe())
print((iso["temp_c"] > 45).sum(), "temperatures above 45")

count    400.00000
mean      52.02950
std      117.00944
min       34.70000
25%       37.00000
50%       37.70000
75%       38.20000
max      999.00000
Name: temp_c, dtype: float64
6 temperatures above 45


In [11]:
iso.loc[~iso["temp_c"].between(30, 45), "temp_c"] = np.nan
iso["temp_c"].describe()

count    394.000000
mean      37.608629
std        0.925784
min       34.700000
25%       36.925000
50%       37.600000
75%       38.200000
max       39.800000
Name: temp_c, dtype: float64

## 7. Missing values

In [12]:
iso.isna().sum()

isolate_id         0
patient_id         0
collected          0
specimen           0
organism           0
ciprofloxacin     24
ceftriaxone       31
mic_cipro          0
temp_c             6
month              0
weekday            0
mic_sign         292
mic_value         60
dtype: int64

Options, depending on why the value is missing:

- leave it missing and report it (often the honest choice)
- drop rows: `dropna(subset=[...])`
- fill: `fillna(value)`, or with the median of a group

Filling is a modelling decision; in the machine learning guide it is done inside a pipeline.

In [13]:
print(len(iso.dropna(subset=["ciprofloxacin"])))
iso["temp_filled"] = iso["temp_c"].fillna(iso.groupby("organism")["temp_c"].transform("median"))
iso[["temp_c", "temp_filled"]].isna().sum()

376


temp_c         6
temp_filled    0
dtype: int64

## 8. Rename and choose types

In [14]:
clean = (
    iso
    .rename(columns={"ciprofloxacin": "cip", "ceftriaxone": "cro"})
    .astype({"organism": "category", "specimen": "category"})
    .drop(columns=["temp_filled"])
)
clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   isolate_id  400 non-null    str           
 1   patient_id  400 non-null    str           
 2   collected   400 non-null    datetime64[us]
 3   specimen    400 non-null    category      
 4   organism    400 non-null    category      
 5   cip         376 non-null    category      
 6   cro         369 non-null    category      
 7   mic_cipro   400 non-null    str           
 8   temp_c      394 non-null    float64       
 9   month       400 non-null    period[M]     
 10  weekday     400 non-null    str           
 11  mic_sign    108 non-null    str           
 12  mic_value   340 non-null    float64       
dtypes: category(4), datetime64[us](1), float64(2), period[M](1), str(5)
memory usage: 30.3 KB


## Write the clean file and keep the steps as code

Never clean by hand in Excel: keep every step in code so it can be rerun when new data arrive.

In [15]:
clean.to_csv("../data/isolates_clean.csv", index=False)
clean.head()

,isolate_id,patient_id,collected,specimen,organism,cip,cro,mic_cipro,temp_c,month,weekday,mic_sign,mic_value
0,ISO-0001,P0114,2025-03-26,Blood,E. coli,R,S,<=0.06,36.8,2025-03,Wednesday,<=,0.06
1,ISO-0002,P0212,2025-04-10,Urine,P. aeruginosa,R,R,2,37.0,2025-04,Thursday,NaN,2.00
2,ISO-0003,P0019,2025-06-24,Blood,E. coli,S,S,0.5,37.6,2025-06,Tuesday,NaN,0.50
3,ISO-0004,P0097,2025-05-19,Wound swab,S. aureus,R,R,0.25,37.9,2025-05,Monday,NaN,0.25
4,ISO-0005,P0038,2025-02-18,Wound swab,K. pneumoniae,S,S,1,38.6,2025-02,Tuesday,NaN,1.00
